In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Bronze: Weather
# MAGIC
# MAGIC **Source:** Flat CSVs at `abfss://raw@.../weather/*.csv`
# MAGIC **Target:** `retaildp.bronze.weather` (managed Delta in bronze container)
# MAGIC **Pattern:** Strict schema → FAILFAST read → 6 quality gates → `MERGE INTO` on
# MAGIC `(obs_date, store_no)`.
# MAGIC
# MAGIC Idempotent: safe to re-run; revised observations overwrite cleanly.
# MAGIC Used downstream by Gold layer for weather-adjusted demand analysis.

# COMMAND ----------

# MAGIC %md
# MAGIC ## 1. Configuration

# COMMAND ----------

CATALOG       = "retaildp"
SCHEMA        = "bronze"
TABLE         = "weather"
TARGET_TABLE  = f"{CATALOG}.{SCHEMA}.{TABLE}"

RAW_BASE         = "abfss://raw@stretaildpsatyaki01.dfs.core.windows.net/"
RAW_WEATHER_PATH = RAW_BASE + "weather/"

VALID_CONDITIONS = {"SUNNY", "CLOUDY", "RAINY", "SNOWY", "STORMY", "PARTLY_CLOUDY"}

spark.sql(f"USE CATALOG {CATALOG}")
print(f"Target table: {TARGET_TABLE}")
print(f"Source path:  {RAW_WEATHER_PATH}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 2. Strict schema (no inference)

# COMMAND ----------

from pyspark.sql.types import (
    StructType, StructField, DateType, StringType, DoubleType, IntegerType,
)

WEATHER_SCHEMA = StructType([
    StructField("obs_date",         DateType(),    nullable=False),
    StructField("store_no",         IntegerType(), nullable=False),
    StructField("city",             StringType(),  nullable=False),
    StructField("country",          StringType(),  nullable=False),
    StructField("temp_max_c",       DoubleType(),  nullable=False),
    StructField("temp_min_c",       DoubleType(),  nullable=False),
    StructField("precipitation_mm", DoubleType(),  nullable=False),
    StructField("condition",        StringType(),  nullable=False),
])

# COMMAND ----------

# MAGIC %md
# MAGIC ## 3. Read raw CSVs

# COMMAND ----------

raw_df = (
    spark.read
        .option("header", "true")
        .option("dateFormat", "yyyy-MM-dd")
        .option("mode", "FAILFAST")
        .schema(WEATHER_SCHEMA)
        .csv(RAW_WEATHER_PATH + "*.csv")
)

raw_count = raw_df.count()
print(f"Read {raw_count:,} rows from {RAW_WEATHER_PATH}")
display(raw_df.limit(5))

# COMMAND ----------

# MAGIC %md
# MAGIC ## 4. Quality gates (assertions — fail loudly before write)

# COMMAND ----------

from pyspark.sql.functions import col, current_date, current_timestamp, lit

# 4.1 No null in key cols (StructType nullable=False catches at parse; double-check here)
null_keys = raw_df.filter(
    col("obs_date").isNull() | col("store_no").isNull() | col("condition").isNull()
).count()
assert null_keys == 0, f"Quality fail: {null_keys} rows with NULL key/condition"

# 4.2 No future-dated observations
future = raw_df.filter(col("obs_date") > current_date()).count()
assert future == 0, f"Quality fail: {future} rows with future obs_date"

# 4.3 Temperature within plausible range (-60°C to +60°C)
bad_temp_range = raw_df.filter(
    (col("temp_max_c") < -60) | (col("temp_max_c") > 60) |
    (col("temp_min_c") < -60) | (col("temp_min_c") > 60)
).count()
assert bad_temp_range == 0, f"Quality fail: {bad_temp_range} rows with implausible temperature"

# 4.4 temp_max_c >= temp_min_c (no inverted readings)
bad_temp_order = raw_df.filter(col("temp_max_c") < col("temp_min_c")).count()
assert bad_temp_order == 0, f"Quality fail: {bad_temp_order} rows with max < min temp"

# 4.5 Precipitation non-negative
bad_precip = raw_df.filter(col("precipitation_mm") < 0).count()
assert bad_precip == 0, f"Quality fail: {bad_precip} rows with negative precipitation"

# 4.6 Condition must be one of the 6 valid codes
bad_condition = raw_df.filter(~col("condition").isin(list(VALID_CONDITIONS))).count()
assert bad_condition == 0, f"Quality fail: {bad_condition} rows with invalid condition code"

print("All 6 quality gates passed.")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 5. Add ingestion metadata and stage

# COMMAND ----------

staged = (
    raw_df
        .withColumn("_ingest_ts",   current_timestamp())
        .withColumn("_source_file", col("_metadata.file_path"))
)

staged.createOrReplaceTempView("weather_stage")
print(f"Staged {staged.count():,} rows with metadata columns.")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 6. Create target table if it doesn't exist

# COMMAND ----------

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TARGET_TABLE} (
    obs_date         DATE     NOT NULL,
    store_no         INT      NOT NULL,
    city             STRING   NOT NULL,
    country          STRING   NOT NULL,
    temp_max_c       DOUBLE   NOT NULL,
    temp_min_c       DOUBLE   NOT NULL,
    precipitation_mm DOUBLE   NOT NULL,
    condition        STRING   NOT NULL,
    _ingest_ts       TIMESTAMP,
    _source_file     STRING
)
USING DELTA
TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.autoOptimize.autoCompact'   = 'true'
)
COMMENT 'Bronze weather observations — daily per-store temperature, precipitation, condition.'
""")

print(f"Target table {TARGET_TABLE} ready.")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 7. MERGE INTO (upsert on (obs_date, store_no))

# COMMAND ----------

merge_result = spark.sql(f"""
MERGE INTO {TARGET_TABLE} AS tgt
USING weather_stage    AS src
   ON tgt.obs_date = src.obs_date
  AND tgt.store_no = src.store_no
WHEN MATCHED THEN UPDATE SET
    tgt.city             = src.city,
    tgt.country          = src.country,
    tgt.temp_max_c       = src.temp_max_c,
    tgt.temp_min_c       = src.temp_min_c,
    tgt.precipitation_mm = src.precipitation_mm,
    tgt.condition        = src.condition,
    tgt._ingest_ts       = src._ingest_ts,
    tgt._source_file     = src._source_file
WHEN NOT MATCHED THEN INSERT (
    obs_date, store_no, city, country,
    temp_max_c, temp_min_c, precipitation_mm, condition,
    _ingest_ts, _source_file
) VALUES (
    src.obs_date, src.store_no, src.city, src.country,
    src.temp_max_c, src.temp_min_c, src.precipitation_mm, src.condition,
    src._ingest_ts, src._source_file
)
""")

display(merge_result)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 8. Sanity summary — by country

# COMMAND ----------

display(spark.sql(f"""
SELECT
    country,
    COUNT(*)                 AS row_count,
    COUNT(DISTINCT store_no) AS store_count,
    MIN(obs_date)            AS first_date,
    MAX(obs_date)            AS last_date,
    ROUND(MIN(temp_min_c), 1) AS coldest_c,
    ROUND(MAX(temp_max_c), 1) AS hottest_c,
    ROUND(AVG(precipitation_mm), 2) AS avg_precip_mm
FROM {TARGET_TABLE}
GROUP BY country
ORDER BY country
"""))

# COMMAND ----------

# MAGIC %md
# MAGIC ## 9. Sanity summary — condition distribution

# COMMAND ----------

display(spark.sql(f"""
SELECT
    condition,
    COUNT(*) AS observation_count,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
FROM {TARGET_TABLE}
GROUP BY condition
ORDER BY observation_count DESC
"""))

# COMMAND ----------

# MAGIC %md
# MAGIC ## 10. Confirm physical location is in the bronze container

# COMMAND ----------

display(spark.sql(f"DESCRIBE EXTENDED {TARGET_TABLE}"))
# Look for the `Location` row — should start with abfss://bronze@stretaildpsatyaki01...